In [ ]:
import cairosvg

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files
import cairosvg


REPO_ID = "lil-lab/kilogram"
REPO_TYPE = "dataset"

OUT_DIR = Path("kilogram_dense10_sample")
SVG_DIR = OUT_DIR / "tangram_svg"
PNG_DIR = OUT_DIR / "tangram_png"

OUT_DIR.mkdir(exist_ok=True)
SVG_DIR.mkdir(exist_ok=True)
PNG_DIR.mkdir(exist_ok=True)


def find_one_file(files, pattern, description):
    """
    Find exactly one file matching a regex pattern, or raise a useful error.
    """
    matches = [f for f in files if re.search(pattern, f)]
    if not matches:
        raise FileNotFoundError(f"Could not find {description}. Pattern: {pattern}")

    # Prefer shorter paths if there are duplicates.
    matches = sorted(matches, key=lambda x: (len(x), x))
    return matches[0]


def get_annotation_text(annotation):
    """
    Robustly pull out whole and part annotations from the KiloGram JSON schema.
    """
    whole = ""
    part_annotations = {}

    if isinstance(annotation, dict):
        # Expected schema from README:
        # annotation["whole"]["wholeAnnotation"]
        if "whole" in annotation and isinstance(annotation["whole"], dict):
            whole = annotation["whole"].get("wholeAnnotation", "")

        # Part annotations correspond to SVG polygon IDs.
        if "part" in annotation and isinstance(annotation["part"], dict):
            part_annotations = annotation["part"]

    return whole, part_annotations


print("Listing files in Hugging Face dataset repo...")
files = list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)

# -----------------------------
# 1. Download dense10.json only
# -----------------------------
dense10_file = find_one_file(
    files,
    pattern=r"(^|/)dense10\.json$",
    description="dense10.json",
)

print(f"Found dense10.json at: {dense10_file}")

dense10_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename=dense10_file,
)

with open(dense10_path, "r", encoding="utf-8") as f:
    dense10 = json.load(f)

print(f"Loaded dense10 with {len(dense10)} tangrams.")


# -----------------------------
# 2. Reformat dense10.json to CSV
# -----------------------------
rows = []

for tangram_name, tangram_obj in dense10.items():
    annotations = tangram_obj.get("annotations", [])

    for annotation_idx, annotation in enumerate(annotations):
        whole_text, part_annotations = get_annotation_text(annotation)

        rows.append({
            "tangram_name": tangram_name,
            "annotation_idx": annotation_idx,
            "whole_annotation": whole_text,
            "part_annotations_json": json.dumps(part_annotations, ensure_ascii=False),
            "num_part_labels": len(part_annotations),
            "svg_filename_expected": f"{tangram_name}.svg",
            "png_filename_expected": f"{tangram_name}.png",
        })

df = pd.DataFrame(rows)

csv_path = OUT_DIR / "dense10_readable.csv"
df.to_csv(csv_path, index=False)

print(f"Saved readable CSV to: {csv_path}")
print(df.head())


# -----------------------------
# 3. Pick first annotation for first 5 shapes
# -----------------------------
first_5_tangrams = list(dense10.keys())[:5]

sample_rows = []

for tangram_name in first_5_tangrams:
    annotations = dense10[tangram_name].get("annotations", [])
    if not annotations:
        print(f"Skipping {tangram_name}: no annotations found.")
        continue

    annotation_idx = 0
    annotation = annotations[annotation_idx]
    whole_text, part_annotations = get_annotation_text(annotation)

    sample_rows.append({
        "tangram_name": tangram_name,
        "annotation_idx": annotation_idx,
        "whole_annotation": whole_text,
        "part_annotations_json": json.dumps(part_annotations, ensure_ascii=False),
    })

sample_df = pd.DataFrame(sample_rows)
sample_csv_path = OUT_DIR / "first_5_first_annotation.csv"
sample_df.to_csv(sample_csv_path, index=False)

print(f"Saved first-5 sample CSV to: {sample_csv_path}")


# -----------------------------
# 4. Download only matching SVG files
# -----------------------------
for tangram_name in first_5_tangrams:
    # Look for exact SVG filename anywhere in repo.
    # Usually this should be something like tangrams-svg/{tangram_name}.svg
    svg_matches = [
        f for f in files
        if f.endswith(f"/{tangram_name}.svg") or f == f"{tangram_name}.svg"
    ]

    if not svg_matches:
        print(f"WARNING: Could not find SVG for {tangram_name}")
        continue

    svg_file = sorted(svg_matches, key=lambda x: (len(x), x))[0]
    print(f"Downloading SVG for {tangram_name}: {svg_file}")

    local_svg_cache_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=svg_file,
    )

    local_svg_path = SVG_DIR / f"{tangram_name}.svg"

    with open(local_svg_cache_path, "rb") as src:
        with open(local_svg_path, "wb") as dst:
            dst.write(src.read())

    # -----------------------------
    # 5. Convert SVG to PNG
    # -----------------------------
    local_png_path = PNG_DIR / f"{tangram_name}.png"

    cairosvg.svg2png(
        url=str(local_svg_path),
        write_to=str(local_png_path),
        output_width=512,
        output_height=512,
    )

    print(f"Saved SVG: {local_svg_path}")
    print(f"Saved PNG: {local_png_path}")


print("\nDone.")
print(f"Main CSV: {csv_path}")
print(f"Sample CSV: {sample_csv_path}")
print(f"SVG folder: {SVG_DIR}")
print(f"PNG folder: {PNG_DIR}")

In [ ]:
from pathlib import Path
import json
import re
import xml.etree.ElementTree as ET

import pandas as pd
import requests
import cairosvg


# -----------------------------
# Configuration
# -----------------------------

DENSE10_URL = "https://raw.githubusercontent.com/lil-lab/kilogram/main/dataset/dense10.json"
GITHUB_TREE_API = "https://api.github.com/repos/lil-lab/kilogram/git/trees/main?recursive=1"
RAW_BASE = "https://raw.githubusercontent.com/lil-lab/kilogram/main"

DATA_DIR = Path("kilogram_test")
SVG_DIR = Path("tangram_svg")
PNG_DIR = Path("tangram_png")

DATA_DIR.mkdir(exist_ok=True)
SVG_DIR.mkdir(exist_ok=True)
PNG_DIR.mkdir(exist_ok=True)


# -----------------------------
# Helpers
# -----------------------------

def download_text(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def download_file(url: str, out_path: Path) -> None:
    response = requests.get(url)
    response.raise_for_status()
    out_path.write_bytes(response.content)


def normalize_tangram_record(tangram_name, tangram_obj):
    """
    dense10.json is expected to look like:
    {
      tangram_name: {
        "snd": ...,
        "pnd": ...,
        "psa": ...,
        "annotations": [...]
      }
    }

    This function flattens each annotation into one CSV row.
    """
    rows = []

    annotations = tangram_obj.get("annotations", [])

    for ann_idx, ann in enumerate(annotations):
        whole = ann.get("whole", {})
        whole_text = whole.get("wholeAnnotation", "")

        part_dict = ann.get("part", {})

        # Part labels by polygon id, e.g. {"1": "head", "2": "body", ...}
        parts_joined = "; ".join(
            f"{piece_id}: {label}"
            for piece_id, label in part_dict.items()
        )

        rows.append({
            "tangram_name": tangram_name,
            "annotation_idx": ann_idx,
            "whole_annotation": whole_text,
            "part_annotations": parts_joined,
            "part_annotations_json": json.dumps(part_dict, ensure_ascii=False),
            "snd": tangram_obj.get("snd"),
            "pnd": tangram_obj.get("pnd"),
            "psa": tangram_obj.get("psa"),
        })

    return rows


def get_github_tree_paths():
    """
    Gets GitHub repo file paths without cloning the repo.
    This downloads metadata only, not the whole dataset.
    """
    response = requests.get(GITHUB_TREE_API)
    response.raise_for_status()
    tree = response.json()["tree"]
    return [item["path"] for item in tree if item["type"] == "blob"]


def find_svg_path(tangram_name: str, repo_paths: list[str]) -> str:
    """
    Find the SVG path for a tangram name.

    Expected file name is usually something like:
    tangrams-svg/page5-207.svg

    But this searches repo paths so the exact folder can vary.
    """
    target_filename = f"{tangram_name}.svg"

    matches = [
        path for path in repo_paths
        if path.endswith(target_filename)
    ]

    if not matches:
        raise FileNotFoundError(f"Could not find SVG for {tangram_name}")

    # Prefer paths that explicitly mention tangrams-svg
    matches = sorted(matches, key=lambda p: ("tangrams-svg" not in p, p))
    return matches[0]


def extract_piece_number(svg_element):
    """
    Try to identify the tangram piece number from an SVG element.

    The SVG polygon ids may be exactly "1", "2", etc.,
    or something like "polygon1", "piece-1", etc.
    """
    element_id = svg_element.attrib.get("id", "")

    if element_id.isdigit():
        return element_id

    match = re.search(r"(\d+)$", element_id)
    if match:
        return match.group(1)

    return None


def set_svg_fill(element, color):
    """
    Set fill color while preserving other style attributes when possible.
    """
    style = element.attrib.get("style", "")

    if style:
        style_parts = []
        found_fill = False

        for item in style.split(";"):
            item = item.strip()
            if not item:
                continue

            if item.startswith("fill:"):
                style_parts.append(f"fill:{color}")
                found_fill = True
            else:
                style_parts.append(item)

        if not found_fill:
            style_parts.append(f"fill:{color}")

        element.set("style", ";".join(style_parts))
    else:
        element.set("fill", color)


def make_colored_annotation_svg(raw_svg_path: Path, colored_svg_path: Path, part_dict: dict):
    """
    Color SVG pieces according to the first annotation.

    If multiple SVG polygons have the same part label, they get the same color.
    """
    colors = [
        "#f94144",  # red
        "#f3722c",  # orange
        "#f9c74f",  # yellow
        "#90be6d",  # green
        "#43aa8b",  # teal
        "#577590",  # blue-gray
        "#9b5de5",  # purple
    ]

    # Assign one color per distinct part label
    unique_labels = []
    for piece_id, label in part_dict.items():
        if label not in unique_labels:
            unique_labels.append(label)

    label_to_color = {
        label: colors[i % len(colors)]
        for i, label in enumerate(unique_labels)
    }

    tree = ET.parse(raw_svg_path)
    root = tree.getroot()

    # XML namespace handling
    namespace_match = re.match(r"\{.*\}", root.tag)
    namespace = namespace_match.group(0) if namespace_match else ""

    # Color polygons and paths if their ids correspond to piece ids
    for element in root.iter():
        tag_without_namespace = element.tag.replace(namespace, "")

        if tag_without_namespace not in {"polygon", "path"}:
            continue

        piece_num = extract_piece_number(element)

        if piece_num is None:
            continue

        if piece_num in part_dict:
            label = part_dict[piece_num]
            color = label_to_color[label]
            set_svg_fill(element, color)

    tree.write(colored_svg_path, encoding="utf-8", xml_declaration=True)


# -----------------------------
# 1. Download dense10.json
# -----------------------------

dense10_path = DATA_DIR / "dense10.json"

print("Downloading dense10.json...")
dense10_text = download_text(DENSE10_URL)
dense10_path.write_text(dense10_text, encoding="utf-8")

with open(dense10_path, "r", encoding="utf-8") as f:
    dense10 = json.load(f)

print(f"Loaded {len(dense10)} tangrams from dense10.json")


# -----------------------------
# 2. Flatten dense10.json to CSV
# -----------------------------

rows = []

for tangram_name, tangram_obj in dense10.items():
    rows.extend(normalize_tangram_record(tangram_name, tangram_obj))

df = pd.DataFrame(rows)

csv_path = DATA_DIR / "dense10_flat.csv"
df.to_csv(csv_path, index=False)

print(f"Saved readable CSV to: {csv_path}")
print(df.head())


# -----------------------------
# 3. Download first 5 SVGs and convert to PNG
# -----------------------------

print("Reading GitHub file tree metadata...")
repo_paths = get_github_tree_paths()

first_5_tangrams = list(dense10.keys())[:5]

selected_rows = []

for tangram_name in first_5_tangrams:
    tangram_obj = dense10[tangram_name]
    first_annotation = tangram_obj["annotations"][0]
    part_dict = first_annotation.get("part", {})

    print(f"\nProcessing {tangram_name}")

    svg_repo_path = find_svg_path(tangram_name, repo_paths)
    svg_url = f"{RAW_BASE}/{svg_repo_path}"

    raw_svg_path = SVG_DIR / f"{tangram_name}.svg"
    colored_svg_path = SVG_DIR / f"{tangram_name}_ann0_colored.svg"
    png_path = PNG_DIR / f"{tangram_name}_ann0_colored.png"

    print(f"Downloading SVG from: {svg_url}")
    download_file(svg_url, raw_svg_path)

    print(f"Creating colored annotation SVG: {colored_svg_path}")
    make_colored_annotation_svg(
        raw_svg_path=raw_svg_path,
        colored_svg_path=colored_svg_path,
        part_dict=part_dict,
    )

    print(f"Converting SVG to PNG: {png_path}")
    cairosvg.svg2png(
        url=str(colored_svg_path),
        write_to=str(png_path),
        output_width=512,
        output_height=512,
    )

    selected_rows.append({
        "tangram_name": tangram_name,
        "annotation_idx": 0,
        "whole_annotation": first_annotation.get("whole", {}).get("wholeAnnotation", ""),
        "part_annotations": "; ".join(f"{k}: {v}" for k, v in part_dict.items()),
        "raw_svg_path": str(raw_svg_path),
        "colored_svg_path": str(colored_svg_path),
        "png_path": str(png_path),
    })


selected_df = pd.DataFrame(selected_rows)
selected_csv_path = DATA_DIR / "first5_selected_annotations.csv"
selected_df.to_csv(selected_csv_path, index=False)

print(f"\nSaved selected examples CSV to: {selected_csv_path}")
print(selected_df)

In [ ]:
from pathlib import Path
import json
import re
import xml.etree.ElementTree as ET

import pandas as pd
import requests
import cairosvg
from tqdm import tqdm


# ============================================================
# Configuration
# ============================================================

DENSE10_URL = "https://raw.githubusercontent.com/lil-lab/kilogram/main/dataset/dense10.json"
GITHUB_TREE_API = "https://api.github.com/repos/lil-lab/kilogram/git/trees/main?recursive=1"
RAW_BASE = "https://raw.githubusercontent.com/lil-lab/kilogram/main"

BASE_DIR = Path("kilogram_dense10_images")
DATA_DIR = BASE_DIR / "data"

RAW_SVG_DIR = BASE_DIR / "tangram_svg" / "raw"
COLORED_SVG_DIR = BASE_DIR / "tangram_svg" / "colored_by_annotation"
RAW_PNG_DIR = BASE_DIR / "tangram_png" / "raw"
COLORED_PNG_DIR = BASE_DIR / "tangram_png" / "colored_by_annotation"

for folder in [DATA_DIR, RAW_SVG_DIR, COLORED_SVG_DIR, RAW_PNG_DIR, COLORED_PNG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# Download helpers
# ============================================================

def download_text(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def download_file(url: str, out_path: Path, overwrite: bool = False) -> None:
    if out_path.exists() and not overwrite:
        return

    response = requests.get(url)
    response.raise_for_status()
    out_path.write_bytes(response.content)


def get_github_tree_paths() -> list[str]:
    """
    Reads GitHub file metadata without cloning the full repo.
    """
    response = requests.get(GITHUB_TREE_API)
    response.raise_for_status()
    tree = response.json()["tree"]
    return [item["path"] for item in tree if item["type"] == "blob"]


def find_svg_path(tangram_name: str, repo_paths: list[str]) -> str:
    """
    Find the SVG file corresponding to a tangram name.
    """
    target = f"{tangram_name}.svg"

    matches = [p for p in repo_paths if p.endswith(target)]

    if not matches:
        raise FileNotFoundError(f"Could not find SVG for tangram: {tangram_name}")

    # Prefer the expected tangrams-svg folder if there are multiple matches.
    matches = sorted(matches, key=lambda p: ("tangrams-svg" not in p, p))
    return matches[0]


# ============================================================
# JSON flattening
# ============================================================

def flatten_dense10(dense10: dict) -> pd.DataFrame:
    rows = []

    for tangram_name, tangram_obj in dense10.items():
        annotations = tangram_obj.get("annotations", [])

        for annotation_idx, ann in enumerate(annotations):
            whole = ann.get("whole", {})
            part_dict = ann.get("part", {})

            part_annotations = "; ".join(
                f"{piece_id}: {label}"
                for piece_id, label in part_dict.items()
            )

            rows.append({
                "tangram_name": tangram_name,
                "annotation_idx": annotation_idx,
                "whole_annotation": whole.get("wholeAnnotation", ""),
                "part_annotations": part_annotations,
                "part_annotations_json": json.dumps(part_dict, ensure_ascii=False),
                "snd": tangram_obj.get("snd"),
                "pnd": tangram_obj.get("pnd"),
                "psa": tangram_obj.get("psa"),
            })

    return pd.DataFrame(rows)


# ============================================================
# SVG coloring helpers
# ============================================================

COLOR_PALETTE = [
    "#f94144",  # red
    "#f9ff89",  # yellow
    "#c12cf3",  # purple
    "#6af400",  # green
    "#f3932c",  # orange
    "#fbc3f5",  # pink
    "#4378ff",  # blue
]


def strip_namespace(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def extract_piece_id(element: ET.Element) -> str | None:
    """
    Try to get a piece id from an SVG element.

    The KiloGram JSON says the part ids correspond to SVG polygon ids.
    This function handles ids like:
      "1"
      "piece-1"
      "polygon1"
    """
    element_id = element.attrib.get("id", "")

    if element_id.isdigit():
        return element_id

    match = re.search(r"(\d+)$", element_id)
    if match:
        return match.group(1)

    return None


def get_style_dict(style_string: str) -> dict:
    style_dict = {}

    for item in style_string.split(";"):
        item = item.strip()
        if not item or ":" not in item:
            continue

        key, value = item.split(":", 1)
        style_dict[key.strip()] = value.strip()

    return style_dict


def style_dict_to_string(style_dict: dict) -> str:
    return ";".join(f"{k}:{v}" for k, v in style_dict.items())


def set_fill(element: ET.Element, color: str) -> None:
    """
    Set fill while preserving existing SVG style information.
    """
    style_string = element.attrib.get("style")

    if style_string:
        style_dict = get_style_dict(style_string)
        style_dict["fill"] = color
        element.set("style", style_dict_to_string(style_dict))
    else:
        element.set("fill", color)


def make_label_to_color(part_dict: dict) -> dict:
    """
    Assign the same color to pieces with the same part label.

    Example:
      {"3": "feet", "4": "feet"} -> pieces 3 and 4 get same color.
    """
    unique_labels = []

    for _, label in part_dict.items():
        if label not in unique_labels:
            unique_labels.append(label)

    return {
        label: COLOR_PALETTE[i % len(COLOR_PALETTE)]
        for i, label in enumerate(unique_labels)
    }


def color_svg_by_annotation(
    raw_svg_path: Path,
    colored_svg_path: Path,
    part_dict: dict,
) -> dict:
    """
    Create an annotation-specific colored SVG.

    Returns a mapping from piece id to color so it can be saved in the index CSV.
    """
    tree = ET.parse(raw_svg_path)
    root = tree.getroot()

    label_to_color = make_label_to_color(part_dict)
    piece_to_color = {}

    for element in root.iter():
        tag = strip_namespace(element.tag)

        # Most likely polygon, but path is included just in case.
        if tag not in {"polygon", "path"}:
            continue

        piece_id = extract_piece_id(element)

        if piece_id is None:
            continue

        if piece_id in part_dict:
            label = part_dict[piece_id]
            color = label_to_color[label]
            set_fill(element, color)
            piece_to_color[piece_id] = color

    tree.write(colored_svg_path, encoding="utf-8", xml_declaration=True)

    return piece_to_color


# ============================================================
# Main pipeline
# ============================================================

print("Downloading dense10.json...")
dense10_path = DATA_DIR / "dense10.json"
dense10_text = download_text(DENSE10_URL)
dense10_path.write_text(dense10_text, encoding="utf-8")

with open(dense10_path, "r", encoding="utf-8") as f:
    dense10 = json.load(f)

print(f"Loaded {len(dense10)} tangrams.")

print("Flattening dense10.json to CSV...")
dense10_df = flatten_dense10(dense10)
dense10_csv_path = DATA_DIR / "dense10_flat.csv"
dense10_df.to_csv(dense10_csv_path, index=False)
print(f"Saved: {dense10_csv_path}")

print("Reading GitHub repo file tree metadata...")
repo_paths = get_github_tree_paths()

image_rows = []

print("Downloading SVGs and generating PNGs...")
for tangram_name, tangram_obj in tqdm(dense10.items(), desc="Tangrams"):
    svg_repo_path = find_svg_path(tangram_name, repo_paths)
    svg_url = f"{RAW_BASE}/{svg_repo_path}"

    raw_svg_path = RAW_SVG_DIR / f"{tangram_name}.svg"
    raw_png_path = RAW_PNG_DIR / f"{tangram_name}.png"

    # Download the original raw SVG once per tangram.
    download_file(svg_url, raw_svg_path)

    # Convert raw SVG to raw PNG once per tangram.
    if not raw_png_path.exists():
        cairosvg.svg2png(
            url=str(raw_svg_path),
            write_to=str(raw_png_path),
            output_width=512,
            output_height=512,
        )

    annotations = tangram_obj.get("annotations", [])

    for annotation_idx, ann in enumerate(annotations):
        whole_text = ann.get("whole", {}).get("wholeAnnotation", "")
        part_dict = ann.get("part", {})

        colored_stem = f"{tangram_name}_ann{annotation_idx:02d}"

        colored_svg_path = COLORED_SVG_DIR / f"{colored_stem}.svg"
        colored_png_path = COLORED_PNG_DIR / f"{colored_stem}.png"

        piece_to_color = color_svg_by_annotation(
            raw_svg_path=raw_svg_path,
            colored_svg_path=colored_svg_path,
            part_dict=part_dict,
        )

        cairosvg.svg2png(
            url=str(colored_svg_path),
            write_to=str(colored_png_path),
            output_width=512,
            output_height=512,
        )

        image_rows.append({
            "tangram_name": tangram_name,
            "annotation_idx": annotation_idx,
            "whole_annotation": whole_text,
            "part_annotations": "; ".join(f"{k}: {v}" for k, v in part_dict.items()),
            "part_annotations_json": json.dumps(part_dict, ensure_ascii=False),
            "piece_to_color_json": json.dumps(piece_to_color, ensure_ascii=False),
            "raw_svg_path": str(raw_svg_path),
            "raw_png_path": str(raw_png_path),
            "colored_svg_path": str(colored_svg_path),
            "colored_png_path": str(colored_png_path),
            "snd": tangram_obj.get("snd"),
            "pnd": tangram_obj.get("pnd"),
            "psa": tangram_obj.get("psa"),
        })


image_index_df = pd.DataFrame(image_rows)
image_index_path = DATA_DIR / "dense10_image_index.csv"
image_index_df.to_csv(image_index_path, index=False)

print("\nDone.")
print(f"Saved flat annotation CSV: {dense10_csv_path}")
print(f"Saved image index CSV: {image_index_path}")
print(f"Raw SVGs: {RAW_SVG_DIR}")
print(f"Raw PNGs: {RAW_PNG_DIR}")
print(f"Colored annotation SVGs: {COLORED_SVG_DIR}")
print(f"Colored annotation PNGs: {COLORED_PNG_DIR}")
print(f"Total annotation-specific images created: {len(image_index_df)}")

In [ ]:
from pathlib import Path
import json
import re
import xml.etree.ElementTree as ET

import pandas as pd
import requests
import cairosvg
from tqdm import tqdm


# ============================================================
# Configuration
# ============================================================

DENSE10_URL = "https://raw.githubusercontent.com/lil-lab/kilogram/main/dataset/dense10.json"
GITHUB_TREE_API = "https://api.github.com/repos/lil-lab/kilogram/git/trees/main?recursive=1"
RAW_BASE = "https://raw.githubusercontent.com/lil-lab/kilogram/main"

BASE_DIR = Path("kilogram_dense10_normalized")
DATA_DIR = BASE_DIR / "data"

RAW_SVG_DIR = BASE_DIR / "tangram_svg" / "raw"
NORM_SVG_DIR = BASE_DIR / "tangram_svg" / "normalized_colored"

RAW_PNG_DIR = BASE_DIR / "tangram_png" / "raw"
NORM_PNG_DIR = BASE_DIR / "tangram_png" / "normalized_colored"

for folder in [DATA_DIR, RAW_SVG_DIR, NORM_SVG_DIR, RAW_PNG_DIR, NORM_PNG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# Your palette
# ============================================================

COLOR_PALETTE = [
    "#f94144",  # red
    "#f9ff89",  # yellow
    "#c12cf3",  # purple
    "#6af400",  # green
    "#f3932c",  # orange
    "#fbc3f5",  # pink
    "#4378ff",  # blue
    "#9d6d43",  # brown
    "#29fbff",  # cyan
    "#c3c3c3",  # gray
]

COLOR_NAME_MAP = {
    "#f94144": "red",
    "#f9ff89": "yellow",
    "#c12cf3": "purple",
    "#6af400": "green",
    "#f3932c": "orange",
    "#fbc3f5": "pink",
    "#4378ff": "blue",
    "#9d6d43": "brown",
    "#29fbff": "cyan",
    "#c3c3c3": "gray",
}


# ============================================================
# Optional normalization map
# ============================================================
# This is where you can decide whether certain labels should
# be treated as the same normalized label.
#
# For example:
#   "feet" -> "foot"
#   "paws" -> "paw"
#   "front legs" -> "front leg"
#
# If you want ONLY exact label matches (after lowercasing and
# whitespace cleanup), keep this minimal.
# ============================================================

NORMALIZATION_MAP = {
    "feet": "foot",
    "paws": "paw",
    "head": "hed",
    "back": "bek",
    # "legs": "leg",
    "front paws": "front paw",
    "back paws": "back paw",
    "front legs": "front leg",
    "back legs": "back leg",
    "hind legs": "hind leg",
    "hind paws": "hind paw",
}


# ============================================================
# Download helpers
# ============================================================

def download_text(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def download_file(url: str, out_path: Path, overwrite: bool = False) -> None:
    if out_path.exists() and not overwrite:
        return
    response = requests.get(url)
    response.raise_for_status()
    out_path.write_bytes(response.content)


def get_github_tree_paths() -> list[str]:
    response = requests.get(GITHUB_TREE_API)
    response.raise_for_status()
    tree = response.json()["tree"]
    return [item["path"] for item in tree if item["type"] == "blob"]


def find_svg_path(tangram_name: str, repo_paths: list[str]) -> str:
    target = f"{tangram_name}.svg"
    matches = [p for p in repo_paths if p.endswith(target)]

    if not matches:
        raise FileNotFoundError(f"Could not find SVG for tangram: {tangram_name}")

    matches = sorted(matches, key=lambda p: ("tangrams-svg" not in p, p))
    return matches[0]


# ============================================================
# Label normalization
# ============================================================

def normalize_label(label: str) -> str:
    """
    Basic normalization:
    - lowercase
    - strip whitespace
    - replace underscores/hyphens with spaces
    - collapse repeated spaces
    - optionally map to a canonical label with NORMALIZATION_MAP
    """
    label = label.lower().strip()
    label = label.replace("_", " ").replace("-", " ")
    label = re.sub(r"\s+", " ", label)

    if label in NORMALIZATION_MAP:
        return NORMALIZATION_MAP[label]

    return label


# ============================================================
# SVG helpers
# ============================================================

def strip_namespace(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def extract_piece_id(element: ET.Element) -> str | None:
    """
    Handles ids like:
      "1"
      "polygon1"
      "piece-1"
    """
    element_id = element.attrib.get("id", "")

    if element_id.isdigit():
        return element_id

    match = re.search(r"(\d+)$", element_id)
    if match:
        return match.group(1)

    return None


def get_style_dict(style_string: str) -> dict:
    style_dict = {}
    for item in style_string.split(";"):
        item = item.strip()
        if not item or ":" not in item:
            continue
        key, value = item.split(":", 1)
        style_dict[key.strip()] = value.strip()
    return style_dict


def style_dict_to_string(style_dict: dict) -> str:
    return ";".join(f"{k}:{v}" for k, v in style_dict.items())


def set_fill(element: ET.Element, color: str) -> None:
    style_string = element.attrib.get("style")
    if style_string:
        style_dict = get_style_dict(style_string)
        style_dict["fill"] = color
        element.set("style", style_dict_to_string(style_dict))
    else:
        element.set("fill", color)


def color_svg_by_piece_labels(
    raw_svg_path: Path,
    out_svg_path: Path,
    piece_to_label: dict,
    label_to_color: dict,
) -> dict:
    """
    Colors each piece according to its normalized label and the
    tangram-level label_to_color mapping.

    Returns a piece_id -> color_hex mapping.
    """
    tree = ET.parse(raw_svg_path)
    root = tree.getroot()

    piece_to_color = {}

    for element in root.iter():
        tag = strip_namespace(element.tag)
        if tag not in {"polygon", "path"}:
            continue

        piece_id = extract_piece_id(element)
        if piece_id is None:
            continue

        if piece_id in piece_to_label:
            norm_label = piece_to_label[piece_id]
            color = label_to_color[norm_label]
            set_fill(element, color)
            piece_to_color[piece_id] = color

    tree.write(out_svg_path, encoding="utf-8", xml_declaration=True)
    return piece_to_color


# ============================================================
# Tangram-level stable color mapping
# ============================================================

def build_tangram_label_to_color(annotations: list[dict]) -> dict:
    """
    Build ONE stable label->color mapping for an entire tangram.

    Strategy:
    1. Read annotation 0 first and assign colors in that order.
    2. Process later annotations. Any new normalized label gets
       the next unused color.

    This gives you the behavior:
    if 'head' is red in annotation 0, then later 'head' stays red.
    """
    label_to_color = {}
    next_color_idx = 0

    for ann in annotations:
        part_dict = ann.get("part", {})
        for piece_id, label in part_dict.items():
            norm_label = normalize_label(label)
            if norm_label not in label_to_color:
                color = COLOR_PALETTE[next_color_idx % len(COLOR_PALETTE)]
                label_to_color[norm_label] = color
                next_color_idx += 1

    return label_to_color


# ============================================================
# Main pipeline
# ============================================================

print("Downloading dense10.json...")
dense10_path = DATA_DIR / "dense10.json"
dense10_text = download_text(DENSE10_URL)
dense10_path.write_text(dense10_text, encoding="utf-8")

with open(dense10_path, "r", encoding="utf-8") as f:
    dense10 = json.load(f)

print(f"Loaded {len(dense10)} tangrams.")

print("Reading GitHub repo metadata...")
repo_paths = get_github_tree_paths()

rows = []

for tangram_name, tangram_obj in tqdm(dense10.items(), desc="Tangrams"):
    annotations = tangram_obj.get("annotations", [])

    # Download raw SVG once per tangram
    svg_repo_path = find_svg_path(tangram_name, repo_paths)
    svg_url = f"{RAW_BASE}/{svg_repo_path}"

    raw_svg_path = RAW_SVG_DIR / f"{tangram_name}.svg"
    raw_png_path = RAW_PNG_DIR / f"{tangram_name}.png"

    download_file(svg_url, raw_svg_path)

    if not raw_png_path.exists():
        cairosvg.svg2png(
            url=str(raw_svg_path),
            write_to=str(raw_png_path),
            output_width=512,
            output_height=512,
        )

    # Build stable label->color mapping for this tangram
    label_to_color = build_tangram_label_to_color(annotations)
    label_to_color_name = {
        label: COLOR_NAME_MAP[color]
        for label, color in label_to_color.items()
    }

    for annotation_idx, ann in enumerate(annotations):
        whole_annotation = ann.get("whole", {}).get("wholeAnnotation", "")
        part_dict = ann.get("part", {})

        # Normalize labels for this annotation
        normalized_part_dict = {
            piece_id: normalize_label(label)
            for piece_id, label in part_dict.items()
        }

        out_stem = f"{tangram_name}_ann{annotation_idx:02d}"
        out_svg_path = NORM_SVG_DIR / f"{out_stem}.svg"
        out_png_path = NORM_PNG_DIR / f"{out_stem}.png"

        piece_to_color_hex = color_svg_by_piece_labels(
            raw_svg_path=raw_svg_path,
            out_svg_path=out_svg_path,
            piece_to_label=normalized_part_dict,
            label_to_color=label_to_color,
        )

        cairosvg.svg2png(
            url=str(out_svg_path),
            write_to=str(out_png_path),
            output_width=512,
            output_height=512,
        )

        piece_to_color_name = {
            piece_id: COLOR_NAME_MAP[color]
            for piece_id, color in piece_to_color_hex.items()
        }

        rows.append({
            "tangram_name": tangram_name,
            "annotation_idx": annotation_idx,
            "whole_annotation": whole_annotation,
            "part_annotations": "; ".join(f"{k}: {v}" for k, v in part_dict.items()),
            "part_annotations_json": json.dumps(part_dict, ensure_ascii=False),

            "normalized_part_annotations_json": json.dumps(normalized_part_dict, ensure_ascii=False),

            "label_to_color_hex_json": json.dumps(label_to_color, ensure_ascii=False),
            "label_to_color_name_json": json.dumps(label_to_color_name, ensure_ascii=False),

            "piece_to_color_hex_json": json.dumps(piece_to_color_hex, ensure_ascii=False),
            "piece_to_color_name_json": json.dumps(piece_to_color_name, ensure_ascii=False),

            "raw_svg_path": str(raw_svg_path),
            "raw_png_path": str(raw_png_path),
            "normalized_svg_path": str(out_svg_path),
            "normalized_png_path": str(out_png_path),

            "snd": tangram_obj.get("snd"),
            "pnd": tangram_obj.get("pnd"),
            "psa": tangram_obj.get("psa"),
        })

# Save CSV
df = pd.DataFrame(rows)
csv_path = DATA_DIR / "dense10_image_index_normalized.csv"
df.to_csv(csv_path, index=False)

print("\nDone.")
print(f"Saved CSV: {csv_path}")
print(f"Raw SVGs: {RAW_SVG_DIR}")
print(f"Raw PNGs: {RAW_PNG_DIR}")
print(f"Normalized colored SVGs: {NORM_SVG_DIR}")
print(f"Normalized colored PNGs: {NORM_PNG_DIR}")
print(f"Total rows: {len(df)}")

In [ ]:
from pathlib import Path
import json
import pandas as pd
import re


# -----------------------------
# Input / output
# -----------------------------

INPUT_CSV = Path("dense10_image_index_with_color_names.csv")
OUTPUT_CSV = Path("dense10_image_index_with_descriptions.csv")


# -----------------------------
# Tangram piece number -> shape
# -----------------------------

PIECE_TO_SHAPE = {
    "1": "square",
    "2": "parallelogram",
    "3": "small triangle",
    "4": "small triangle",
    "5": "medium triangle",
    "6": "large triangle",
    "7": "large triangle",
}


# -----------------------------
# Small grammar helpers
# -----------------------------

ARTICLE_WORDS = {"a", "an", "the"}


IRREGULAR_PLURALS = {
    "people", "children", "men", "women", "teeth", "feet", "mice", "geese",
    "oxen", "cacti", "fungi", "nuclei", "alumni", "data"
}


SINGULAR_S_ENDINGS = {
    # common singular words ending in -s that should still get "a/an"
    "bus", "class", "glass", "dress", "boss", "cross", "grass",
    "octopus", "cactus", "hippopotamus", "rhinoceros"
}


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def starts_with_article(phrase):
    phrase = clean_text(phrase).lower()
    words = phrase.split()
    first = words[0] if words else ""
    return first in ARTICLE_WORDS


def get_first_content_word(phrase):
    phrase = clean_text(phrase).lower()
    words = phrase.split()

    if not words:
        return ""

    if words[0] in ARTICLE_WORDS and len(words) > 1:
        return words[1]

    return words[0]


def get_head_word(phrase):
    """
    Heuristic: use the last token as the likely noun head.
    Examples:
      "pyramids" -> pyramids
      "stack of books" -> books
      "person sitting" -> sitting  # imperfect but okay for simple labels
    """
    phrase = clean_text(phrase).lower()
    phrase = re.sub(r"[^\w\s/-]", "", phrase)
    words = phrase.split()

    if not words:
        return ""

    return words[-1]


def looks_plural(phrase):
    """
    Simple heuristic for whole_annotation labels:
      pyramids -> plural
      birds -> plural
      people -> plural
      chair -> singular
      bus -> singular
    """
    phrase = clean_text(phrase).lower()

    if starts_with_article(phrase):
        return False

    head = get_head_word(phrase)

    if not head:
        return False

    if head in IRREGULAR_PLURALS:
        return True

    if head in SINGULAR_S_ENDINGS:
        return False

    # words ending in ies are usually plural: "bunnies", "fairies"
    if head.endswith("ies") and len(head) > 3:
        return True

    # words ending in s are often plural, but avoid ss/us cases where possible
    if head.endswith("s") and not head.endswith(("ss", "us")):
        return True

    return False


def choose_article(phrase):
    phrase = clean_text(phrase)

    if not phrase:
        return "a"

    first_word = get_first_content_word(phrase)
    return "an" if first_word and first_word[0] in "aeiou" else "a"


def with_article(phrase):
    """
    Returns:
      chair -> a chair
      anvil -> an anvil
      pyramids -> pyramids
      a chair -> a chair
      the dog -> the dog
    """
    phrase = clean_text(phrase)

    if not phrase:
        return ""

    if starts_with_article(phrase):
        return phrase

    if looks_plural(phrase):
        return phrase

    return f"{choose_article(phrase)} {phrase}"


def without_leading_article(phrase):
    """
    Returns:
      a chair -> chair
      an anvil -> anvil
      the dog -> dog
      pyramids -> pyramids
    """
    phrase = clean_text(phrase)
    words = phrase.split()

    if words and words[0].lower() in ARTICLE_WORDS:
        return " ".join(words[1:])

    return phrase

# ARTICLE_WORDS = {"a", "an", "the"}


# def clean_text(x):
#     if pd.isna(x):
#         return ""
#     return str(x).strip()


# def starts_with_article(phrase):
#     phrase = clean_text(phrase).lower()
#     first = phrase.split()[0] if phrase.split() else ""
#     return first in ARTICLE_WORDS


# def choose_article(phrase):
#     """
#     Basic a/an choice.
#     This is intentionally simple. It handles most cases like:
#       an anvil
#       a chair
#       a bull
#     """
#     phrase = clean_text(phrase)
#     if not phrase:
#         return "a"

#     first_word = phrase.split()[0].lower()
#     return "an" if first_word[0] in "aeiou" else "a"


# def with_article(phrase):
#     """
#     Returns:
#       chair -> a chair
#       anvil -> an anvil
#       a chair -> a chair
#       the dog -> the dog
#     """
#     phrase = clean_text(phrase)

#     if not phrase:
#         return ""

#     if starts_with_article(phrase):
#         return phrase

#     return f"{choose_article(phrase)} {phrase}"


# def without_leading_article(phrase):
#     """
#     Returns:
#       a chair -> chair
#       an anvil -> anvil
#       the dog -> dog
#       chair -> chair

#     Useful for:
#       The base of the chair...
#     instead of:
#       The base of the a chair...
#     """
#     phrase = clean_text(phrase)
#     words = phrase.split()

#     if words and words[0].lower() in ARTICLE_WORDS:
#         return " ".join(words[1:])

#     return phrase


def shape_with_article(shape):
    """
    Returns:
      square -> a square
      parallelogram -> a parallelogram
      small triangle -> a small triangle
    """
    shape = clean_text(shape)
    return f"{choose_article(shape)} {shape}"


def comma_join(items):
    """
    Joins:
      ["a"] -> "a"
      ["a", "b"] -> "a and b"
      ["a", "b", "c"] -> "a, b, and c"
    """
    items = [clean_text(x) for x in items if clean_text(x)]

    if not items:
        return ""

    if len(items) == 1:
        return items[0]

    if len(items) == 2:
        return f"{items[0]} and {items[1]}"

    return f"{', '.join(items[:-1])}, and {items[-1]}"


def sentence_join(part_sentences):
    """
    Joins sentence fragments like:
      ["The base ...", "the backrest ...", "the seat ..."]
    into one sentence.
    """
    parts = [clean_text(x).rstrip(".") for x in part_sentences if clean_text(x)]

    if not parts:
        return ""

    if len(parts) == 1:
        return parts[0] + "."

    if len(parts) == 2:
        return f"{parts[0]}, and {parts[1]}."

    return f"{', '.join(parts[:-1])}, and {parts[-1]}."


# -----------------------------
# JSON / parsing helpers
# -----------------------------

def parse_json_cell(cell):
    if pd.isna(cell) or not str(cell).strip():
        return {}

    try:
        return json.loads(cell)
    except json.JSONDecodeError:
        return {}


def parse_part_annotations_order(part_annotations):
    """
    Parses:
      "6: barrel; 4: head; 2: rump"

    Returns ordered list:
      [("6", "barrel"), ("4", "head"), ("2", "rump")]
    """
    part_annotations = clean_text(part_annotations)

    if not part_annotations:
        return []

    parsed = []

    for item in part_annotations.split(";"):
        item = item.strip()

        if not item or ":" not in item:
            continue

        piece_id, label = item.split(":", 1)
        parsed.append((piece_id.strip(), label.strip()))

    return parsed


def group_parts_in_annotation_order(part_annotations, part_json, color_name_json):
    """
    Groups pieces by part label in the order the labels first appear.

    Example:
      3: base; 2: base; 6: base; 7: backrest

    Becomes:
      [
        {
          "label": "base",
          "pieces": ["3", "2", "6"],
          "colors": ["red"],
          "color": "red",
          "shapes": ["small triangle", "parallelogram", "large triangle"]
        },
        ...
      ]
    """
    ordered_piece_labels = parse_part_annotations_order(part_annotations)

    # Fallback to JSON if part_annotations string is missing
    if not ordered_piece_labels and part_json:
        ordered_piece_labels = list(part_json.items())

    grouped = []
    label_to_group = {}

    for piece_id, label in ordered_piece_labels:
        piece_id = str(piece_id)
        label = clean_text(label)

        if not label:
            continue

        if label not in label_to_group:
            group = {
                "label": label,
                "pieces": [],
                "colors": [],
                "shapes": [],
            }
            grouped.append(group)
            label_to_group[label] = group

        group = label_to_group[label]
        group["pieces"].append(piece_id)

        color = color_name_json.get(piece_id, "")
        shape = PIECE_TO_SHAPE.get(piece_id, "")

        if color:
            group["colors"].append(color)

        if shape:
            group["shapes"].append(shape)

    for group in grouped:
        unique_colors = []
        for color in group["colors"]:
            if color not in unique_colors:
                unique_colors.append(color)

        group["unique_colors"] = unique_colors
        group["color"] = unique_colors[0] if len(unique_colors) == 1 else comma_join(unique_colors)

    return grouped


def build_color_to_pieces(color_name_json):
    color_to_pieces = {}

    for piece_id, color in color_name_json.items():
        piece_id = str(piece_id)
        color = clean_text(color)

        if not color:
            continue

        color_to_pieces.setdefault(color, []).append(piece_id)

    return color_to_pieces


# -----------------------------
# Description builders
# -----------------------------

def build_photo_description(whole_annotation):
    whole = with_article(whole_annotation)

    if not whole:
        return ""

    return f"The image shows {whole}."


def build_part_description(group, whole_object, is_first=False):
    label = group["label"]
    color = group["color"]

    if not label or not color:
        return ""

    be = be_verb_for_label(label)

    if is_first:
        return f"The {label} of the {whole_object} {be} {color}"

    return f"the {label} {be} {color}"


def build_shape_description(group, whole_object, color_to_pieces, is_first=False):
    label = group["label"]
    color = group["color"]
    shapes = group["shapes"]

    if not label or not color or not shapes:
        return "", False

    be = be_verb_for_label(label)

    unique_colors = group.get("unique_colors", [])
    group_pieces = set(group["pieces"])

    is_single_shape_color = False

    if len(unique_colors) == 1:
        color_pieces = set(color_to_pieces.get(unique_colors[0], []))
        is_single_shape_color = len(color_pieces) == 1 and color_pieces == group_pieces

    # Single-piece, single-color case:
    # singular: "The head of the bull is a small yellow triangle."
    # plural should almost never happen here, but we handle it safely.
    if is_single_shape_color and len(shapes) == 1 and len(unique_colors) == 1:
        shape = shapes[0]
        color = unique_colors[0]

        if "triangle" in shape:
            size = shape.replace(" triangle", "")
            shape_phrase = f"a {size} {color} triangle"
        else:
            shape_phrase = f"a {color} {shape}"

        if is_first:
            desc = f"The {label} of the {whole_object} {be} {shape_phrase}"
        else:
            desc = f"the {label} {be} {shape_phrase}"

        return desc, is_single_shape_color

    # Multi-piece case:
    # singular: "The base ... is red and is comprised of..."
    # plural: "The legs ... are yellow and are comprised of..."
    shape_list = comma_join([shape_with_article(shape) for shape in shapes])

    if is_first:
        desc = (
            f"The {label} of the {whole_object} {be} {color} "
            f"and {be} comprised of {shape_list}"
        )
    else:
        desc = (
            f"the {label} {be} {color} "
            f"and {be} comprised of {shape_list}"
        )

    return desc, is_single_shape_color


# -----------------------------
# Main row function
# -----------------------------

def add_descriptions_to_row(row):
    whole_annotation = clean_text(row.get("whole_annotation", ""))
    whole_object = without_leading_article(whole_annotation)

    part_json = parse_json_cell(row.get("part_annotations_json", ""))
    color_name_json = parse_json_cell(row.get("piece_to_color_name_json", ""))

    groups = group_parts_in_annotation_order(
        part_annotations=row.get("part_annotations", ""),
        part_json=part_json,
        color_name_json=color_name_json,
    )

    color_to_pieces = build_color_to_pieces(color_name_json)

    output = {}

    # Photo description
    output["photo_description"] = build_photo_description(whole_annotation)

    # Initialize empty columns
    for i in range(1, 8):
        output[f"part_description_{i}"] = ""
        output[f"part_description_shape_{i}"] = ""
        output[f"part_description_shape_{i}_single_shape_color"] = ""

    # Part descriptions
    part_descs = []
    shape_descs = []

    for i, group in enumerate(groups[:7], start=1):
        part_desc = build_part_description(
            group=group,
            whole_object=whole_object,
            is_first=(i == 1),
        )

        shape_desc, single_shape_color_flag = build_shape_description(
            group=group,
            whole_object=whole_object,
            color_to_pieces=color_to_pieces,
            is_first=(i == 1),
        )

        output[f"part_description_{i}"] = part_desc + "." if part_desc else ""
        output[f"part_description_shape_{i}"] = shape_desc + "." if shape_desc else ""
        output[f"part_description_shape_{i}_single_shape_color"] = single_shape_color_flag

        if part_desc:
            part_descs.append(part_desc)

        if shape_desc:
            shape_descs.append(shape_desc)

    output["part_description"] = sentence_join(part_descs)
    output["part_description_shape"] = sentence_join(shape_descs)

    return pd.Series(output)

IRREGULAR_PLURAL_PARTS = {
    "feet", "teeth", "people", "children", "men", "women", "mice", "geese"
}

SINGULAR_S_ENDING_PARTS = {
    "bus", "class", "glass", "dress", "grass", "cross", "boss"
}


def looks_plural_label(label):
    """
    Heuristic for part labels:
      leg -> singular
      legs -> plural
      feet -> plural
      back legs -> plural
      front paws -> plural
      leg/chest -> singular-ish unless the final item is plural
    """
    label = clean_text(label).lower()

    if not label:
        return False

    # For labels like "leg/chest", check the last slash-separated part.
    label = label.split("/")[-1].strip()

    # Use last word as the head.
    words = re.sub(r"[^\w\s-]", "", label).split()
    if not words:
        return False

    head = words[-1]

    if head in IRREGULAR_PLURAL_PARTS:
        return True

    if head in SINGULAR_S_ENDING_PARTS:
        return False

    if head.endswith("ies") and len(head) > 3:
        return True

    if head.endswith("s") and not head.endswith(("ss", "us")):
        return True

    return False


def be_verb_for_label(label):
    return "are" if looks_plural_label(label) else "is"

# -----------------------------
# Run
# -----------------------------

df = pd.read_csv(INPUT_CSV)

new_cols = df.apply(add_descriptions_to_row, axis=1)

df_out = pd.concat([df, new_cols], axis=1)

df_out.to_csv(OUTPUT_CSV, index=False)

print(f"Saved updated CSV to: {OUTPUT_CSV}")
print(f"Rows: {len(df_out)}")
print("Added columns:")
for col in new_cols.columns:
    print(f"  - {col}")